# Pandas Titanic Cookbook

A self-contained cookbook based on the attached pandas cheat sheet and the supplied `titanic.csv` dataset.

The same cookbook is also provided as a Python script. Place `titanic.csv` in the same directory before running either file.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = Path('titanic.csv')
OUTPUT_DIR = Path('titanic_cookbook_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_columns', 30)
pd.set_option('display.precision', 3)

## Load and standardize the Titanic data

In [ ]:
titanic = pd.read_csv(CSV_PATH)
titanic.columns = titanic.columns.str.strip()
titanic['Name'] = titanic['Name'].str.strip()
for col in ['Ticket', 'Cabin', 'Embarked']:
    titanic[col] = titanic[col].astype('string').str.strip()

display(titanic.head())
print('Shape:', titanic.shape)
display(titanic.dtypes)
display(titanic.isna().sum().sort_values(ascending=False))

## Creating features

In [ ]:
titanic = titanic.assign(
    FamilySize=lambda d: d['SibSp'] + d['Parch'] + 1,
    IsAlone=lambda d: (d['SibSp'] + d['Parch'] == 0).astype('int8'),
    HasCabin=lambda d: d['Cabin'].notna().astype('int8'),
    Title=lambda d: d['Name'].str.extract(r',\s*([^.]*)\.', expand=False).str.strip(),
    FarePerPerson=lambda d: d['Fare'] / (d['SibSp'] + d['Parch'] + 1),
    AgeBand=lambda d: pd.cut(d['Age'], bins=[-np.inf, 12, 18, 35, 60, np.inf], labels=['child', 'adolescent', 'adult', 'middle_age', 'senior'])
)
display(titanic[['Name', 'FamilySize', 'IsAlone', 'HasCabin', 'Title', 'FarePerPerson', 'AgeBand']].head())

## Selecting and filtering

In [ ]:
display(titanic.loc[titanic['Sex'].eq('female'), ['PassengerId', 'Name', 'Pclass', 'Age', 'Survived']].head())
display(titanic.iloc[10:15])
display(titanic.filter(regex=r'^(Passenger|Surv|Fare|Age)').head())
display(titanic.query("Sex == 'female' and Pclass <= 2")[[ 'Name', 'Sex', 'Pclass', 'Survived']].head())
display(titanic[titanic['Embarked'].isin(['C', 'Q'])][['Name', 'Embarked']].head())

## Reshaping, sorting, renaming, and concatenation

In [ ]:
summary_long = (titanic.groupby('Pclass', as_index=False)[['Survived', 'Fare']].mean()
                .melt(id_vars='Pclass', var_name='variable', value_name='value')
                .rename(columns={'variable': 'metric', 'value': 'mean_value'})
                .query('mean_value >= 0'))
display(summary_long)

survival_wide = (titanic.groupby(['Pclass', 'Sex'], as_index=False)['Survived'].mean()
                .pivot(index='Pclass', columns='Sex', values='Survived'))
display(survival_wide)

display(titanic.sort_values('Fare', ascending=False)[['Name', 'Pclass', 'Fare']].head())
display(titanic.rename(columns={'Pclass': 'PassengerClass'}).head())
display(pd.concat([titanic.iloc[:3, :5], titanic.iloc[3:6, :5]], ignore_index=True))

## Combining data sets

In [ ]:
ticket_summary = titanic.groupby('Ticket', as_index=False).agg(
    TicketPassengers=('PassengerId', 'size'),
    TicketMeanFare=('Fare', 'mean')
)
merged = pd.merge(titanic, ticket_summary, how='left', on='Ticket')
display(merged[['Name', 'Ticket', 'TicketPassengers', 'TicketMeanFare']].head())

ticket_metadata = pd.DataFrame({
    'Ticket': titanic['Ticket'].dropna().drop_duplicates().head(10),
    'TicketCategory': 'observed'
)
outer = pd.merge(titanic[['PassengerId', 'Ticket']].head(10), ticket_metadata, how='outer', on='Ticket', indicator=True)
display(outer)

## Grouping, summarizing, and missing data

In [ ]:
grouped = (titanic.groupby(['Pclass', 'Sex'])
           .agg(Passengers=('PassengerId', 'size'), SurvivalRate=('Survived', 'mean'), MeanAge=('Age', 'mean'), MedianFare=('Fare', 'median'))
           .reset_index())
display(grouped)
display(titanic['Embarked'].value_counts(dropna=False))
display(titanic.nlargest(5, 'Fare')[['Name', 'Fare']])

filled = titanic.assign(
    AgeFilled=titanic['Age'].fillna(titanic['Age'].median()),
    EmbarkedFilled=titanic['Embarked'].fillna(titanic['Embarked'].mode().iat[0]),
    CabinKnown=titanic['Cabin'].notna()
)
display(filled[['Name', 'Age', 'AgeFilled', 'Embarked', 'EmbarkedFilled', 'CabinKnown']].head())

## Ranking, windows, strings, and datetimes

In [ ]:
windowed = titanic.sort_values('PassengerId').copy()
windowed['FareRank'] = windowed['Fare'].rank(method='dense', ascending=False)
windowed['FareShifted'] = windowed['Fare'].shift(1)
windowed['FareCumulative'] = windowed['Fare'].fillna(0).cumsum()
windowed['FareRollingMean'] = windowed['Fare'].rolling(5, min_periods=1).mean()
windowed['Surname'] = windowed['Name'].str.partition(',')[0].str.title()
windowed['TicketHasPeriod'] = windowed['Ticket'].str.contains(r'\.', regex=True, na=False)
windowed['SyntheticBookingDate'] = pd.date_range('1912-04-01', periods=len(windowed), freq='D')
windowed['BookingYear'] = windowed['SyntheticBookingDate'].dt.year
windowed['BookingMonth'] = windowed['SyntheticBookingDate'].dt.month
windowed['BookingDay'] = windowed['SyntheticBookingDate'].dt.day
display(windowed[['PassengerId', 'Fare', 'FareRank', 'FareShifted', 'FareCumulative', 'FareRollingMean', 'Surname', 'BookingDay']].head(10))

## Plotting

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
titanic.groupby('Pclass')['Survived'].mean().plot.bar(ax=axes[0], title='Survival by class', ylim=(0, 1), rot=0)
titanic['Fare'].plot.hist(ax=axes[1], bins=30, title='Fare distribution')
titanic.plot.scatter(x='Age', y='Fare', c='Survived', cmap='coolwarm', alpha=0.55, ax=axes[2], title='Age versus fare')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'titanic_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## Input/output and exercises

In [ ]:
titanic.to_csv(OUTPUT_DIR / 'titanic_featured.csv', index=False)
print('CSV written to', OUTPUT_DIR / 'titanic_featured.csv')

# Exercises:
# 1. Which embarkation port has the highest survival rate?
# 2. Compare survival by Pclass and Sex using a pivot table.
# 3. Extract titles from Name and count them.
# 4. Find passengers travelling alone who paid above the median fare.
# 5. Create three Fare quantiles with pd.qcut and compare survival.
# 6. Save a compact table with PassengerId, Survived, Pclass, Sex, AgeBand, and FarePerPerson.